In [1]:
import pandas as pd
import numpy as np
import os

## Read data

In [2]:
file_to_read = "SPUU_60min.csv"
aggregate_timestamp = "60min"

df = pd.read_csv(f'../backtesting-lib/data/uncleaned/{file_to_read}')

In [3]:
df.head()

,timestamp,open,high,low,close,volume
0,2014-05-28 09:00:00,39.96,40.03,39.96,40.03,200
1,2014-05-28 10:00:00,40.00,40.00,39.86,39.90,900
2,2014-05-28 11:00:00,39.96,40.05,39.96,40.02,900
3,2014-05-28 12:00:00,40.02,40.10,40.00,40.03,11900
4,2014-05-28 13:00:00,40.03,40.14,40.02,40.14,1800


## Parse Timestamps

In [4]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["timestamp"] = df["timestamp"].dt.tz_localize(
    "America/New_York",
    nonexistent="shift_forward",  # handles spring-forward DST
    ambiguous="NaT",  # handles fall-back DST
)
df["timestamp"] = df["timestamp"] + pd.Timedelta(hours=1)
df = df.set_index("timestamp").sort_index()

In [5]:
df.head()

,open,high,low,close,volume
timestamp,,,,,
2014-05-28 10:00:00-04:00,39.96,40.03,39.96,40.03,200
2014-05-28 11:00:00-04:00,40.00,40.00,39.86,39.90,900
2014-05-28 12:00:00-04:00,39.96,40.05,39.96,40.02,900
2014-05-28 13:00:00-04:00,40.02,40.10,40.00,40.03,11900
2014-05-28 14:00:00-04:00,40.03,40.14,40.02,40.14,1800


## Remove After-Hours Data

In [6]:
market_open = "09:30"
market_close = "16:00"

df = df.between_time(market_open, market_close)

In [7]:
df.iloc[:10]

,open,high,low,close,volume
timestamp,,,,,
2014-05-28 10:00:00-04:00,39.96,40.03,39.96,40.03,200
2014-05-28 11:00:00-04:00,40.00,40.00,39.86,39.90,900
2014-05-28 12:00:00-04:00,39.96,40.05,39.96,40.02,900
2014-05-28 13:00:00-04:00,40.02,40.10,40.00,40.03,11900
2014-05-28 14:00:00-04:00,40.03,40.14,40.02,40.14,1800
2014-05-28 15:00:00-04:00,40.12,40.14,40.04,40.09,2400
2014-05-28 16:00:00-04:00,40.08,40.13,39.96,39.96,2700
2014-05-29 10:00:00-04:00,40.12,40.17,40.12,40.12,900
2014-05-29 11:00:00-04:00,40.15,40.15,39.98,40.09,6300


## Fix Stock Splits

In [8]:
POSSIBLE_RATIOS = np.array([1 / 2, 1 / 3, 1 / 4, 1 / 5, 1 / 10, 2 / 1, 3 / 1, 4 / 1, 5 / 1])
TOLERANCE = 0.02 # 2%

def find_nearest_ratio(x):
    i = np.argmin(abs(POSSIBLE_RATIOS - x))
    if abs(POSSIBLE_RATIOS[i] - x) < TOLERANCE:
        return POSSIBLE_RATIOS[i]
    else:
        return np.nan


df["ratio"] = df["open"] / df["open"].shift(1)
df["split_ratio"] = df["ratio"].apply(find_nearest_ratio)

splits = df.dropna(subset=["split_ratio"])  # does not modify original df
if not splits.empty:
    print(f"Detected {len(splits)} split(s):")
    print(splits[["split_ratio"]])

# Adjust historical prices and volumes for each split
df_adjusted = df.copy()
for timestamp, row in splits.iterrows():
    ratio = row["split_ratio"]
    # All rows *before* the split timestamp need to be adjusted
    df_adjusted.loc[df_adjusted.index < timestamp, ["open", "high", "low", "close"]] *= ratio
    df_adjusted.loc[df_adjusted.index < timestamp, "volume"] /= ratio

# Clean up temporary columns
df = df_adjusted.drop(columns=["ratio", "split_ratio"], errors="ignore")

Detected 1 split(s):
                           split_ratio
timestamp                             
2014-06-06 10:00:00-04:00          0.5


## Remove Bad Ticks

In [9]:
# Example thresholds
lower_thresh = 0.5  # 50% lower than neighbors
upper_thresh = 1.5  # 50% higher than neighbors

# Shifted neighbors
prev = df.shift(1)
next = df.shift(-1)

# Initialize mask for bad ticks
bad_mask = pd.Series(False, index=df.index)

# Check each OHLC column
for col in ["open", "high", "low", "close"]:
    bad_col = ((df[col] > prev[col] * upper_thresh) & (df[col] > next[col] * upper_thresh)) | \
              ((df[col] < prev[col] * lower_thresh) & (df[col] < next[col] * lower_thresh))
    bad_mask |= bad_col  # flag if any column is bad

# Logging
num_bad = bad_mask.sum()
print(f"Detected {num_bad} bad tick(s) out of {len(df)} rows")
if num_bad > 0:
    print("Timestamps of bad ticks:")
    print(df.index[bad_mask])

# Remove bad ticks
df = df[~bad_mask]

Detected 0 bad tick(s) out of 15386 rows


## Reindex Data

In [10]:
# Define constants
start_time = "10:00" if aggregate_timestamp == "60min" else market_open
freq = "1h" if aggregate_timestamp == "60min" else aggregate_timestamp

# Get unique dates
dates = df.index.normalize().unique()  # just the dates

# Build full RTH index
full_index = pd.DatetimeIndex(
    [ts for date in dates for ts in pd.date_range(
        start=f"{date.date()} {start_time}",
        end=f"{date.date()} {market_close}",
        freq=freq
    )]
)

# Assign timezone
full_index = full_index.tz_localize("America/New_York", ambiguous="NaT", nonexistent="shift_forward")

# Reindex only RTH
df = df.reindex(full_index)
df.iloc[:15]

,open,high,low,close,volume
2014-05-28 10:00:00-04:00,19.980,20.015,19.980,20.015,400.0
2014-05-28 11:00:00-04:00,20.000,20.000,19.930,19.950,1800.0
2014-05-28 12:00:00-04:00,19.980,20.025,19.980,20.010,1800.0
2014-05-28 13:00:00-04:00,20.010,20.050,20.000,20.015,23800.0
2014-05-28 14:00:00-04:00,20.015,20.070,20.010,20.070,3600.0
2014-05-28 15:00:00-04:00,20.060,20.070,20.020,20.045,4800.0
2014-05-28 16:00:00-04:00,20.040,20.065,19.980,19.980,5400.0
2014-05-29 10:00:00-04:00,20.060,20.085,20.060,20.060,1800.0
2014-05-29 11:00:00-04:00,20.075,20.075,19.990,20.045,12600.0
2014-05-29 12:00:00-04:00,20.040,20.070,20.040,20.045,3400.0


## Linearly Interpolate Missing Data

In [11]:
# Boolean mask: True for rows with at least one missing value
rows_with_missing = df.isna().any(axis=1)

# Count how many rows have missing values
num_missing_rows = rows_with_missing.sum()

# Total number of rows
total_rows = len(df)

# Percentage of rows with missing values
percent_missing = 100 * num_missing_rows / total_rows

print(f"{num_missing_rows} out of {total_rows} rows are missing some value "
      f"({percent_missing:.2f}%)")

df = df.interpolate(method="linear")

4298 out of 19684 rows are missing some value (21.83%)


In [12]:
df.head()

,open,high,low,close,volume
2014-05-28 10:00:00-04:00,19.980,20.015,19.98,20.015,400.0
2014-05-28 11:00:00-04:00,20.000,20.000,19.93,19.950,1800.0
2014-05-28 12:00:00-04:00,19.980,20.025,19.98,20.010,1800.0
2014-05-28 13:00:00-04:00,20.010,20.050,20.00,20.015,23800.0
2014-05-28 14:00:00-04:00,20.015,20.070,20.01,20.070,3600.0


In [13]:
output_dir = "../backtesting-lib/data/cleaned"
os.makedirs(output_dir, exist_ok=True)

# Round to cents and integer volume
df[["open", "high", "low", "close"]] = df[["open", "high", "low", "close"]].round(2)  # cents
df["volume"] = df["volume"].round(0).astype(int)

# Save cleaned data
df.to_csv(f'{output_dir}/{file_to_read}', index=True)